<a href="https://colab.research.google.com/github/CassieMarie0728/colab-notebooks/blob/main/RVC_Feature_Extraction_Colab_UPDATED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVC Feature Extraction — Colab Notebook

Updated smooth-run version.

This notebook is **part 2** of the RVC dataset pipeline. It expects the clean dataset zip from the first notebook, with something like:

```text
cass_rvc_dataset_clean/
├── logs/
│   └── 0_gt_wavs/
├── filelist.txt
└── reports/
```

What this notebook does:
- uploads or reads your prepared zip
- extracts the dataset safely
- finds `logs/0_gt_wavs` automatically
- extracts WORLD F0 files into `logs/2a_f0`
- extracts coarse pitch into `logs/2b-f0nsf`
- extracts HuBERT content features into `logs/3_feature768`
- validates counts and file paths
- creates both a simple filelist and an RVC-style full filelist
- packages the finished dataset zip

The big fix: this version **does not use fairseq**. The older notebook tried to load a Hugging Face Transformers HuBERT checkpoint through fairseq, which is a wonderful way to make Colab eat gravel and die dramatically. This one uses `transformers.HubertModel` directly.


## 1) Install dependencies

This avoids reinstalling CUDA Torch unless absolutely necessary, because Colab already has a GPU-compatible Torch stack most of the time. Reinstalling Torch for no reason is how notebooks start doing occult bullshit.

In [ ]:
import sys, subprocess, os

!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Keep this lightweight. Do not force-reinstall torch/torchaudio unless Colab is missing them.
!pip -q install -U pip setuptools wheel
!pip -q install numpy scipy pandas soundfile librosa pyworld huggingface_hub transformers accelerate tqdm safetensors

# Quick dependency smoke test
import importlib
for pkg in ["numpy", "pandas", "soundfile", "librosa", "pyworld", "torch", "transformers"]:
    importlib.import_module(pkg)
print("Dependencies imported successfully. Colab did not burst into flames.")


## 2) Imports and device check

In [ ]:
import os
import re
import json
import math
import shutil
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import pyworld as pw
import torch
from transformers import HubertModel

from tqdm.auto import tqdm
from google.colab import files

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

if DEVICE != "cuda":
    print("WARNING: No GPU detected. This may still run, but HuBERT feature extraction will be slower than cold molasses.")
else:
    print("GPU:", torch.cuda.get_device_name(0))


## 3) Configuration

Usually you only need to touch `DATASET_ZIP_PATH` if your zip is already in Google Drive. Leave it blank to use the upload button.

For RVC V2-style models, `3_feature768` is the normal target. The audio can stay 40 kHz in `0_gt_wavs`; HuBERT extraction resamples internally to 16 kHz.

In [ ]:
# Leave blank to upload manually with the Colab file picker.
# Example Drive path: "/content/drive/MyDrive/RVC/cass_rvc_dataset_clean.zip"
DATASET_ZIP_PATH = ""

DATASET_NAME = "cass_rvc_dataset_clean"
SPEAKER_ID = 0

# Your prepared wavs are 40 kHz. HuBERT will resample to 16 kHz internally.
AUDIO_LOAD_SR = 40000
HUBERT_INPUT_SR = 16000

# Standard-ish RVC pitch extraction settings.
# F0 is extracted from a 16 kHz version with a 160-sample hop = 10 ms.
F0_EXTRACT_SR = 16000
F0_HOP_LENGTH = 160
F0_FLOOR = 50
F0_CEIL = 1100

# HuBERT settings. Layer 9 is commonly used in RVC-style pipelines.
HUBERT_MODEL_ID = "facebook/hubert-base-ls960"
HUBERT_LAYER = 9
FEATURE_DIR_NAME = "3_feature768"

# Safety / convenience
OVERWRITE_OUTPUTS = True
PACKAGE_OUTPUT_ZIP = True
OUTPUT_ZIP_NAME = "rvc_extracted_dataset"

print(json.dumps({
    "DATASET_ZIP_PATH": DATASET_ZIP_PATH or "manual upload",
    "DATASET_NAME": DATASET_NAME,
    "DEVICE": DEVICE,
    "AUDIO_LOAD_SR": AUDIO_LOAD_SR,
    "HUBERT_INPUT_SR": HUBERT_INPUT_SR,
    "F0_EXTRACT_SR": F0_EXTRACT_SR,
    "F0_HOP_LENGTH": F0_HOP_LENGTH,
    "HUBERT_MODEL_ID": HUBERT_MODEL_ID,
    "HUBERT_LAYER": HUBERT_LAYER,
    "FEATURE_DIR_NAME": FEATURE_DIR_NAME,
}, indent=2))


## 4) Upload or locate your prepared dataset zip

In [ ]:
def resolve_dataset_zip():
    if DATASET_ZIP_PATH:
        p = Path(DATASET_ZIP_PATH)
        assert p.exists(), f"DATASET_ZIP_PATH does not exist: {p}"
        assert p.suffix.lower() == ".zip", f"DATASET_ZIP_PATH must be a zip file: {p}"
        return p

    uploaded = files.upload()
    zip_candidates = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
    assert zip_candidates, "No ZIP uploaded. Upload the prepared dataset ZIP from part 1."

    zip_name = zip_candidates[0]
    zip_path = Path("/content") / zip_name
    with open(zip_path, "wb") as f:
        f.write(uploaded[zip_name])
    return zip_path

zip_path = resolve_dataset_zip()
print("Using ZIP:", zip_path)


## 5) Extract dataset and locate `logs/0_gt_wavs`

In [ ]:
BASE_WORK = Path("/content/rvc_extract_work")
if BASE_WORK.exists():
    shutil.rmtree(BASE_WORK)
BASE_WORK.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    bad = zf.testzip()
    assert bad is None, f"Zip appears corrupted at: {bad}"
    zf.extractall(BASE_WORK)

print("Extracted to:", BASE_WORK)
print("Top-level extracted items:")
for p in sorted(BASE_WORK.iterdir()):
    print(" -", p)

def find_gt_wavs_dirs(base: Path):
    return [p for p in base.rglob("0_gt_wavs") if p.is_dir()]

candidates = find_gt_wavs_dirs(BASE_WORK)
assert candidates, "Could not find logs/0_gt_wavs in extracted dataset. Wrong zip, wrong folder structure, or goblin sabotage."

# Choose the candidate with the most wav files.
gt_wavs_dir = max(candidates, key=lambda d: len(list(d.glob("*.wav"))))
wav_files = sorted(gt_wavs_dir.glob("*.wav"))
assert wav_files, f"Found {gt_wavs_dir}, but it contains no WAV files."

# Expected: dataset_root/logs/0_gt_wavs
logs_dir = gt_wavs_dir.parent
dataset_root = logs_dir.parent
reports_dir = dataset_root / "reports"
f0_dir = logs_dir / "2a_f0"
f0nsf_dir = logs_dir / "2b-f0nsf"
feature_dir = logs_dir / FEATURE_DIR_NAME

output_dirs = [f0_dir, f0nsf_dir, feature_dir, reports_dir]
if OVERWRITE_OUTPUTS:
    for d in [f0_dir, f0nsf_dir, feature_dir]:
        if d.exists():
            shutil.rmtree(d)

for d in output_dirs:
    d.mkdir(parents=True, exist_ok=True)

print("Dataset root:", dataset_root)
print("Logs dir:", logs_dir)
print("GT WAVS dir:", gt_wavs_dir)
print("Found clips:", len(wav_files))


## 6) Quick audio sanity check

This confirms the extracted training clips look like the clean dataset we already built: mono-ish, readable WAV files, sane sample rates, and no cursed empty audio.

In [ ]:
audio_rows = []
for wav_path in tqdm(wav_files, desc="Checking WAVs"):
    try:
        info = sf.info(str(wav_path))
        audio_rows.append({
            "file": wav_path.name,
            "samplerate": info.samplerate,
            "channels": info.channels,
            "duration_sec": round(info.duration, 3),
            "frames": info.frames,
            "subtype": info.subtype,
            "status": "ok",
        })
    except Exception as e:
        audio_rows.append({
            "file": wav_path.name,
            "samplerate": None,
            "channels": None,
            "duration_sec": 0,
            "frames": 0,
            "subtype": None,
            "status": f"error: {e}",
        })

df_audio = pd.DataFrame(audio_rows)
display(df_audio.head())
print("Clips:", len(df_audio))
print("Total duration minutes:", round(df_audio["duration_sec"].sum() / 60, 2))
print("Sample rates:", sorted(df_audio["samplerate"].dropna().unique().tolist()))
print("Statuses:")
print(df_audio["status"].value_counts())

assert (df_audio["status"] == "ok").all(), "Some WAV files could not be read. Check df_audio."
assert df_audio["duration_sec"].sum() > 60, "Total audio is suspiciously short."


## 7) Load HuBERT model

In [ ]:
hubert_model = HubertModel.from_pretrained(HUBERT_MODEL_ID)
hubert_model.to(DEVICE)
hubert_model.eval()

print("HuBERT loaded:", HUBERT_MODEL_ID)
print("Hidden size:", hubert_model.config.hidden_size)
print("Encoder layers:", hubert_model.config.num_hidden_layers)


## 8) Helper functions

In [ ]:
def safe_stem(name):
    stem = Path(name).stem
    stem = re.sub(r"[^a-zA-Z0-9_\-]+", "_", stem)
    stem = re.sub(r"_+", "_", stem).strip("_")
    return stem or "audio"

def save_npy(path, arr):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, arr)

def load_wav_mono(path, sr=None):
    y, actual_sr = librosa.load(path, sr=sr, mono=True)
    return y.astype(np.float32), (sr or actual_sr)

def compute_f0_world_for_file(path):
    y, sr = load_wav_mono(path, sr=F0_EXTRACT_SR)
    y64 = y.astype(np.float64)
    frame_period = 1000 * F0_HOP_LENGTH / F0_EXTRACT_SR
    raw_f0, t = pw.harvest(
        y64, F0_EXTRACT_SR,
        f0_floor=F0_FLOOR,
        f0_ceil=F0_CEIL,
        frame_period=frame_period,
    )
    f0 = pw.stonemask(y64, raw_f0, t, F0_EXTRACT_SR)
    return f0.astype(np.float32)

def coarse_f0(f0):
    f0 = f0.copy()
    f0_mel = 1127 * np.log1p(f0 / 700)
    f0_mel_min = 1127 * np.log1p(F0_FLOOR / 700)
    f0_mel_max = 1127 * np.log1p(F0_CEIL / 700)
    voiced = f0_mel > 0
    f0_mel[voiced] = (f0_mel[voiced] - f0_mel_min) * 254 / (f0_mel_max - f0_mel_min) + 1
    f0_mel[f0_mel <= 1] = 1
    f0_mel[f0_mel > 255] = 255
    return np.rint(f0_mel).astype(np.int16)

@torch.no_grad()
def extract_hubert_features_for_file(path):
    # HuBERT expects 16 kHz. Keep this independent from the stored WAV sample rate.
    y16, _ = load_wav_mono(path, sr=HUBERT_INPUT_SR)
    wav = torch.from_numpy(y16).float().unsqueeze(0).to(DEVICE)
    attention_mask = torch.ones_like(wav, dtype=torch.long, device=DEVICE)

    out = hubert_model(
        wav,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True,
    )

    hidden_states = out.hidden_states
    if hidden_states is not None and len(hidden_states) > HUBERT_LAYER:
        feats = hidden_states[HUBERT_LAYER]
    else:
        feats = out.last_hidden_state

    feats = feats.squeeze(0).detach().float().cpu().numpy().astype(np.float32)
    return feats

def abs_posix(p):
    return Path(p).resolve().as_posix()


## 9) Run F0 extraction

In [ ]:
f0_rows = []

for wav_path in tqdm(wav_files, desc="Extracting F0"):
    stem = safe_stem(wav_path.name)
    try:
        f0 = compute_f0_world_for_file(wav_path)
        f0c = coarse_f0(f0)

        save_npy(f0_dir / f"{stem}.npy", f0)
        save_npy(f0nsf_dir / f"{stem}.npy", f0c)

        voiced = int((f0 > 0).sum())
        f0_rows.append({
            "file": wav_path.name,
            "stem": stem,
            "frames": int(len(f0)),
            "voiced_frames": voiced,
            "voiced_ratio": round(voiced / len(f0), 4) if len(f0) else 0,
            "f0_min": round(float(f0[f0 > 0].min()), 2) if voiced else None,
            "f0_median": round(float(np.median(f0[f0 > 0])), 2) if voiced else None,
            "f0_max": round(float(f0[f0 > 0].max()), 2) if voiced else None,
            "status": "ok",
        })
    except Exception as e:
        f0_rows.append({
            "file": wav_path.name,
            "stem": stem,
            "frames": 0,
            "voiced_frames": 0,
            "voiced_ratio": 0,
            "f0_min": None,
            "f0_median": None,
            "f0_max": None,
            "status": f"error: {e}",
        })

df_f0 = pd.DataFrame(f0_rows)
display(df_f0.head())
print(df_f0["status"].value_counts())


## 10) Run HuBERT feature extraction

In [ ]:
feature_rows = []

for wav_path in tqdm(wav_files, desc="Extracting HuBERT features"):
    stem = safe_stem(wav_path.name)
    try:
        feats = extract_hubert_features_for_file(wav_path)
        save_npy(feature_dir / f"{stem}.npy", feats)

        feature_rows.append({
            "file": wav_path.name,
            "stem": stem,
            "frames": int(feats.shape[0]) if feats.ndim >= 1 else 0,
            "dim": int(feats.shape[1]) if feats.ndim == 2 else None,
            "status": "ok",
        })
    except Exception as e:
        feature_rows.append({
            "file": wav_path.name,
            "stem": stem,
            "frames": 0,
            "dim": None,
            "status": f"error: {e}",
        })

df_feat = pd.DataFrame(feature_rows)
display(df_feat.head())
print(df_feat["status"].value_counts())
print("Feature dims:", sorted(df_feat["dim"].dropna().unique().tolist()))


## 11) Build filelists

This writes two filelists:

- `filelist.txt`: simple `logs/0_gt_wavs/file.wav|speaker_id` style
- `filelist_rvc_full.txt`: full absolute-path RVC-style entries: `wav|feature|f0|f0nsf|speaker_id`

Different RVC forks are annoyingly inconsistent, because apparently naming standards are illegal now.

In [ ]:
simple_filelist_path = dataset_root / "filelist.txt"
full_filelist_path = dataset_root / "filelist_rvc_full.txt"

valid_entries = []
missing_rows = []

for wav_path in sorted(wav_files):
    stem = safe_stem(wav_path.name)
    feat_path = feature_dir / f"{stem}.npy"
    f0_path = f0_dir / f"{stem}.npy"
    f0nsf_path = f0nsf_dir / f"{stem}.npy"

    missing = [str(p) for p in [feat_path, f0_path, f0nsf_path] if not p.exists()]
    if missing:
        missing_rows.append({"file": wav_path.name, "missing": "; ".join(missing)})
        continue

    valid_entries.append((wav_path, feat_path, f0_path, f0nsf_path))

with open(simple_filelist_path, "w", encoding="utf-8") as f:
    for wav_path, feat_path, f0_path, f0nsf_path in valid_entries:
        rel = wav_path.relative_to(dataset_root).as_posix()
        f.write(f"{rel}|{SPEAKER_ID}\n")

with open(full_filelist_path, "w", encoding="utf-8") as f:
    for wav_path, feat_path, f0_path, f0nsf_path in valid_entries:
        f.write(f"{abs_posix(wav_path)}|{abs_posix(feat_path)}|{abs_posix(f0_path)}|{abs_posix(f0nsf_path)}|{SPEAKER_ID}\n")

df_missing = pd.DataFrame(missing_rows)
print("Simple filelist:", simple_filelist_path)
print("Full RVC filelist:", full_filelist_path)
print("Valid entries:", len(valid_entries), "/", len(wav_files))

if len(missing_rows):
    display(df_missing.head(20))
    raise RuntimeError("Some files are missing generated features or F0 files. See df_missing.")

print("Preview full filelist:")
print("\n".join(full_filelist_path.read_text(encoding="utf-8").splitlines()[:3]))


## 12) Validation summary

In [ ]:
feature_count = len(list(feature_dir.glob("*.npy")))
f0_count = len(list(f0_dir.glob("*.npy")))
f0nsf_count = len(list(f0nsf_dir.glob("*.npy")))
filelist_count = len(simple_filelist_path.read_text(encoding="utf-8").splitlines())

summary = {
    "dataset_root": dataset_root.as_posix(),
    "gt_wavs_dir": gt_wavs_dir.as_posix(),
    "feature_dir": feature_dir.as_posix(),
    "wav_files": len(wav_files),
    "f0_count": f0_count,
    "f0nsf_count": f0nsf_count,
    "feature_count": feature_count,
    "filelist_entries": filelist_count,
    "audio_duration_minutes": round(float(df_audio["duration_sec"].sum() / 60), 3),
    "device": DEVICE,
    "feature_model": HUBERT_MODEL_ID,
    "feature_layer": HUBERT_LAYER,
}

display(pd.DataFrame([summary]))

expected = len(wav_files)
assert f0_count == expected, f"F0 count mismatch: {f0_count} vs {expected}"
assert f0nsf_count == expected, f"F0NSF count mismatch: {f0nsf_count} vs {expected}"
assert feature_count == expected, f"Feature count mismatch: {feature_count} vs {expected}"
assert filelist_count == expected, f"Filelist count mismatch: {filelist_count} vs {expected}"

print("Validation passed. The feature extraction goblin has been successfully bullied into compliance.")


## 13) Save reports

In [ ]:
reports_dir.mkdir(parents=True, exist_ok=True)

df_audio.to_csv(reports_dir / "audio_input_report.csv", index=False)
df_f0.to_csv(reports_dir / "f0_report.csv", index=False)
df_feat.to_csv(reports_dir / "feature_report.csv", index=False)

with open(reports_dir / "extraction_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved reports to:", reports_dir)
for p in sorted(reports_dir.glob("*")):
    print(" -", p.name)


## 14) Package the finished dataset

In [ ]:
if PACKAGE_OUTPUT_ZIP:
    zip_base = Path("/content") / OUTPUT_ZIP_NAME
    finished_zip = shutil.make_archive(str(zip_base), "zip", root_dir=dataset_root)
    print("Created:", finished_zip)
else:
    finished_zip = None
    print("PACKAGE_OUTPUT_ZIP is False. Skipping zip packaging.")


## 15) Download the finished ZIP

In [ ]:
if finished_zip:
    files.download(finished_zip)
else:
    print("No zip was created, so there is nothing to download.")


## Output structure

```text
cass_rvc_dataset_clean/
├── logs/
│   ├── 0_gt_wavs/
│   ├── 2a_f0/
│   ├── 2b-f0nsf/
│   └── 3_feature768/
├── reports/
│   ├── audio_input_report.csv
│   ├── f0_report.csv
│   ├── feature_report.csv
│   └── extraction_summary.json
├── filelist.txt
└── filelist_rvc_full.txt
```

Use `logs/0_gt_wavs`, `logs/2a_f0`, `logs/2b-f0nsf`, and `logs/3_feature768` for the next RVC training stage. If your trainer asks for a filelist, try `filelist_rvc_full.txt` first if it expects feature and pitch paths, or `filelist.txt` if it expects the simpler two-column format.
